# Todo #6 — 特征工程 FE-3（StratifiedKFold + OOF Isolation Forest）

> 镜像 [`credit-fraud-feature-engineering-2.ipynb`](credit-fraud-feature-engineering-2.ipynb)；**差异：** 用 **OOF Isolation Forest** 替代 Isolation Forest；CV 仍为 **StratifiedKFold**。

**目标：** 降 FP（1 EUR 优先）> 降 FN；AUC-PR ablation 定稿 `MODEL_FEATURES_V2`。
**导出：** `MODEL_FEATURES_V2_stratifiedif.json`



## 0. 环境与数据加载



In [31]:
# --- 0. 环境与数据加载 ---
# 镜像主 notebook；读入 creditcard.csv 并记录全局欺诈率。
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)

DATA_PATH = '../input/creditcard.csv'


def read_creditcard_csv(path: str) -> pd.DataFrame:
    """依次尝试 utf-8 / 容错 utf-8 / latin-1，避免 UnicodeDecodeError。"""
    for kwargs in (
        {'encoding': 'utf-8'},
        {'encoding': 'utf-8', 'encoding_errors': 'replace'},
        {'encoding': 'latin-1'},
    ):
        try:
            return pd.read_csv(path, **kwargs)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1, 'failed to decode creditcard.csv')


df = read_creditcard_csv(DATA_PATH)
FRAUD_RATE = df['Class'].mean()
print(f'行数: {len(df):,}  |  欺诈笔数: {df["Class"].sum()}  |  欺诈率: {FRAUD_RATE:.4f}')


行数: 284,807  |  欺诈笔数: 492  |  欺诈率: 0.0017


## 1. 建模工具（镜像主 notebook 2.0）



In [32]:
# --- 2.0 1.7 特征构造 + 可复用验证工具（精简镜像）---
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
)
import lightgbm as lgb
import xgboost as xgb

V_COLS = [c for c in df.columns if c.startswith('V')]
BASE_FEATURES = V_COLS + ['Amount', 'Time']

EDA_FEATURES = [
    'log1p_amount',
    'hours_since_start',
    'is_micro_testing',
    'is_one_euro',
    'is_amount_1_30',    # (1, 30] EUR — 难样本/PCA 易混带
    'is_amount_75_110',  # [75, 110] EUR — 难样本/PCA 易混带
]

EDA_FEATURE_DOC = {
    'log1p_amount': 'log1p(Amount)',
    'hours_since_start': 'Time // 3600',
    'is_micro_testing': 'Amount < 1',
    'is_one_euro': 'Amount == 1',
    'is_amount_1_30': '1 < Amount <= 30',
    'is_amount_75_110': '75 <= Amount <= 110',
}
AMOUNT_BAND_FEATURES = ('is_amount_1_30', 'is_amount_75_110')

EARLY_STOPPING_ROUNDS = 50
MAX_BOOST_ROUNDS = 1500
ES_FRAC = 0.25  # 早停监控集占当前 train 折比例（75% fit / 25% ES）
DEFAULT_CLASSIFICATION_THRESHOLD = 0.5
TOP_V_K = 6  # Family A/B 门控交叉用的 Top-V 个数
CV_N_SPLITS = 5           # 组合矩阵 / ablation 可选 CV 折数
CV_RANDOM_STATE = 42


def build_eda_features(data: pd.DataFrame) -> pd.DataFrame:
    """构造 EDA 衍生特征（难样本金额带版；无 is_inactive / is_small_testing）。"""
    out = data.copy()
    out['log1p_amount'] = np.log1p(out['Amount'])
    out['hours_since_start'] = (out['Time'] // 3600).astype(int)
    out['is_micro_testing'] = out['Amount'] < 1
    out['is_one_euro'] = out['Amount'] == 1.0
    out['is_amount_1_30'] = (out['Amount'] > 1) & (out['Amount'] <= 30)
    out['is_amount_75_110'] = (out['Amount'] >= 75) & (out['Amount'] <= 110)
    return out


def _split_early_stop_set(X_tr, y_tr, es_frac=ES_FRAC, random_state=42):
    """切分出用来验证早停的数据"""
    return train_test_split(
        X_tr, y_tr, test_size=es_frac, random_state=random_state, stratify=y_tr,
    )


def make_classifier(model_name: str, y_train: pd.Series, params: dict | None = None):
    """构造树（logloss + 类别不平衡权重）"""
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight='balanced', random_state=42, verbose=-1,
        )
        defaults.update(params)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=42, eval_metric='logloss', verbosity=0,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    return xgb.XGBClassifier(**defaults)


def fit_classifier(clf, model_name: str, X_tr, y_tr, X_es=None, y_es=None):
    """训练（logloss + 早停）"""
    if X_es is None or y_es is None:
        clf.fit(X_tr, y_tr)
        return clf
    if model_name == 'LightGBM':
        clf.fit(
            X_tr, y_tr, eval_set=[(X_es, y_es)], eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
    else:
        clf.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
    return clf


def eval_classifier(
    model_name: str, X_train, y_train, X_val, y_val,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD, random_state: int = 42,
) -> dict:
    """不交叉验证，直接训练评估模型"""
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    clf = make_classifier(model_name, y_fit)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
    proba = clf.predict_proba(X_val)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    return {
        '模型': model_name, '特征数': X_train.shape[1],
        'AUC-PR': average_precision_score(y_val, proba),
        f'F1@{threshold}': f1_score(y_val, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y_val, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y_val, pred, zero_division=0),
        'FP': fp, 'FN': fn, 'proba': proba, 'pred': pred, 'threshold': threshold,
    }


def cross_val_auc_pr(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> tuple[float, float, list[float]]:
    """StratifiedKFold：返回 (AUC-PR 均值, 标准差, 各折分数列表)。"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_scores: list[float] = []
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        fold_scores.append(float(average_precision_score(y_va, clf.predict_proba(X_va)[:, 1])))
    arr = np.array(fold_scores)
    return float(arr.mean()), float(arr.std(ddof=0)), fold_scores


def get_oof_proba(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> np.ndarray:
    """5-fold OOF 概率，用于 CV 下汇总 FP/FN（@threshold）。"""
    oof = np.zeros(len(y), dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        oof[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]
    return oof


def cross_val_eval(
    model_name: str,
    data: pd.DataFrame,
    feature_cols: list,
    y_col: str = 'Class',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """CV 评估：一轮 StratifiedKFold 同时算各折 AUC-PR 与 OOF 概率（避免重复训练）。"""
    X = data[feature_cols]
    y = data[y_col]
    oof_proba = np.zeros(len(y), dtype=float)
    fold_scores: list[float] = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        proba_va = clf.predict_proba(X_va)[:, 1]
        fold_scores.append(float(average_precision_score(y_va, proba_va)))
        oof_proba[va_idx] = proba_va

    arr = np.array(fold_scores)
    pred = (oof_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        '模型': model_name,
        '特征数': len(feature_cols),
        'AUC-PR_mean': float(arr.mean()),
        'AUC-PR_std': float(arr.std(ddof=0)),
        'fold_AUC-PR': fold_scores,
        f'F1@{threshold}': f1_score(y, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y, pred, zero_division=0),
        'FP': int(fp),
        'FN': int(fn),
    }


def feature_ablation(
    data: pd.DataFrame, candidate_features=None, base_features=None,
    model_name: str = 'LightGBM', test_size: float = 0.2, random_state: int = 42,
    include_all_combo: bool = True,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> pd.DataFrame:
    """特征消融；并列输出 FP / FN（@threshold）。"""
    if base_features is None:
        base_features = BASE_FEATURES
    candidate_features = list(candidate_features or [])
    y = data['Class']
    X_train, X_val, y_train, y_val = train_test_split(
        data[base_features], y, test_size=test_size, random_state=random_state, stratify=y,
    )
    val_idx = y_val.index
    specs = [('基线', base_features)]
    for f in candidate_features:
        specs.append((f'+{f}', base_features + [f]))
    if include_all_combo and candidate_features:
        specs.append(('+全部候选', base_features + candidate_features))
    rows, base_ap = [], None
    for label, cols in specs:
        missing = [c for c in cols if c not in data.columns]
        if missing:
            raise KeyError(f'缺少列: {missing}')
        res = eval_classifier(
            model_name, data.loc[X_train.index, cols], y_train,
            data.loc[val_idx, cols], y_val, threshold=threshold,
        )
        if base_ap is None:
            base_ap = res['AUC-PR']
        rows.append({
            '特征集': label, '特征数': res['特征数'], 'AUC-PR': res['AUC-PR'],
            f"F1@{threshold}": res[f'F1@{threshold}'],
            f"Precision@{threshold}": res[f'Precision@{threshold}'],
            f"Recall@{threshold}": res[f'Recall@{threshold}'],
            'FP': res['FP'], 'FN': res['FN'],
            'Δ AUC-PR': res['AUC-PR'] - base_ap,
        })
    return pd.DataFrame(rows).sort_values('AUC-PR', ascending=False).reset_index(drop=True)


df_model = build_eda_features(df)
print('衍生特征已构造；ablation 基线 BASE_FEATURES：')
print(BASE_FEATURES)
print('\nEDA 列（难样本金额带版）：')
for col in EDA_FEATURES:
    print(f"  {col:20s}  {EDA_FEATURE_DOC[col]}")
for col in AMOUNT_BAND_FEATURES:
    n = int(df_model[col].sum())
    print(f"  → {col}: {n:,} 笔 ({n / len(df_model):.2%})")


衍生特征已构造；ablation 基线 BASE_FEATURES：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time']

EDA 列（难样本金额带版）：
  log1p_amount          log1p(Amount)
  hours_since_start     Time // 3600
  is_micro_testing      Amount < 1
  is_one_euro           Amount == 1
  is_amount_1_30        1 < Amount <= 30
  is_amount_75_110      75 <= Amount <= 110
  → is_amount_1_30: 130,588 笔 (45.85%)
  → is_amount_75_110: 20,464 笔 (7.19%)


## 6.0 基础设施：Top-V 选择、门控交叉、OOF Isolation Forest



In [33]:
# =============================================================================
# 6.0 特征工程基础设施
# -----------------------------------------------------------------------------
# 本 cell 提供三类能力：
#   1. pick_top_v_features  — 用树模型 importance 选出与欺诈最相关的 Top-K 个 V 列
#   2. build_cross_features — 构造「？ × V」交叉特征（Family A/B 共用）
#   3. oof_if_anomaly_score — OOF Isolation Forest，产出无泄露的异常分 if_oof_score
# =============================================================================

# --- 依赖导入 ---
# #region agent log
import inspect as _inspect

def _dbg_log(hypothesis_id, location, message, data=None, run_id="pre-fix"):
    import json as _json, time as _time
    payload = {
        "sessionId": "db9af8",
        "runId": run_id,
        "hypothesisId": hypothesis_id,
        "location": location,
        "message": message,
        "data": data or {},
        "timestamp": int(_time.time() * 1000),
    }
    with open("/Users/jingyuhe/.cursor/debug-logs/debug-db9af8.log", "a", encoding="utf-8") as _f:
        _f.write(_json.dumps(payload, ensure_ascii=False) + "\n")
# #endregion

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- Isolation Forest 超参数（可按训练时间 / 效果微调）---
IF_INPUT_COLS = V_COLS          # IF 只吃 V1–V28（PCA 分量），不含 Amount/Time
IF_N_ESTIMATORS = 200           # 孤立树棵数
IF_MAX_SAMPLES = 0.5            # 每棵树子采样比例
IF_CONTAMINATION = 'auto'       # 污染率上界（影响 score 偏移）
IF_MAX_NORMAL_SAMPLES = 50_000  # 每折 normal 子采样上限
IF_RANDOM_STATE = 42            # 控制 KFold 划分、子采样、IF random_state

OOF_IF_IMPL_VERSION = "isolation_forest_v1"  # 用于检测内核是否加载旧版 AE 函数


def pick_top_v_features(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    k: int = TOP_V_K,
    model_name: str = 'LightGBM',
    random_state: int = 42,
) -> list:
    """
    在指定特征集上训一版树分类器，按 feature_importances_（gain）取 Top-K 个 V 列。

      - feature_cols: list | None  → Python 3.10+ 联合类型；None 时用 BASE_FEATURES

    为何需要：Family A 交叉项是 is_one_euro × V_i，试图解决FP误报，不能对 28 个 V 全交叉（维数爆炸），
    先用模型 importance 筛出与欺诈最相关的 V，再与 is_one_euro 相乘。
    """
    feature_cols = feature_cols or BASE_FEATURES  # `or`：None / 空列表时回退默认
    y = data['Class']
    X = data[feature_cols]

    # stratify=y：分层抽样，保证 train/val 欺诈率与全表一致（极不平衡数据必做）
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )

    clf = make_classifier(model_name, y_train)
    # 再从 X_train 切 ES_FRAC（25%）作早停监控集（与 2.0 eval_classifier 一致）
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)

    # feature_importances_ 与 feature_cols 一一对应；只保留 V 列并降序
    imp = pd.Series(clf.feature_importances_, index=feature_cols)
    v_imp = imp[V_COLS].sort_values(ascending=False)
    top_v = list(v_imp.head(k).index)  # .head(k) 取前 k 个 index，转 list
    print(f'Top-{k} V（{model_name} gain）：{top_v}')
    return top_v


def build_cross_features(
    data: pd.DataFrame,
    top_v: list,
    gate_col: str = 'is_one_euro',
    prefix: str = 'one_euro',
) -> tuple[pd.DataFrame, list]:
    """
    特征交叉：new_col = gate_col × V_i。

      - 返回 tuple[pd.DataFrame, list]：新 DataFrame + 新列名列表（供 ablation 传入）

    业务含义：one_euro_Vi 等同于告诉树：「这一维 Vi 的信息，请只在 1 EUR 子空间里用。」
    帮助树模型区分「1 EUR 正常探测」与「1 EUR 欺诈」在 V 空间上的差异（降 FP）。
    否则一棵树更可能在全表上学：Vi > 某阈值 → 更像欺诈
    这个规则会作用在 所有金额 上，包括大量非 1 EUR 交易
    在 1 EUR 上，正常探测和欺诈的 Vi 可能重叠很大，全局阈值就容易误伤正常 1 EUR 交易（FP）
    """
    out = data.copy()            
    gate = out[gate_col].astype(float)  # bool → 0.0/1.0，便于与连续 V 相乘
    new_cols = []
    for v in top_v:
        name = f'{prefix}_{v}'  # 如 one_euro_V14
        out[name] = gate * out[v]   # 逐元素相乘（Series × Series）
        new_cols.append(name)
    return out, new_cols


def oof_if_anomaly_score(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    y_col: str = 'Class',
    n_splits: int = 5,
    random_state: int = IF_RANDOM_STATE,
) -> np.ndarray:
    """
    Out-of-Fold Isolation Forest 异常分（越大越异常）。

    核心防泄露设计：
      - 每折 IF 只在 train 折的 **正常样本** 上训练（学「正常分布」）
      - 异常分只算 **val 折** 样本 → 该样本从未参与该折 IF 的训练
      - 禁止用 in-sample 分数当特征（否则训练集上会偏乐观）

    返回值：长度 = len(data) 的 np.ndarray，与 data 行序对齐，可直接 df['if_oof_score'] = ...
    """
    feature_cols = feature_cols or IF_INPUT_COLS
    X = data[feature_cols].values.astype(np.float64)
    y = data[y_col].values
    oof_score = np.zeros(len(y), dtype=np.float64)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        normal_tr = tr_idx[y[tr_idx] == 0]            # 训练 IF 时剔除欺诈，只学 normal

        # 子采样：normal 过多时随机抽 IF_MAX_NORMAL_SAMPLES 条，加速且通常足够
        if len(normal_tr) > IF_MAX_NORMAL_SAMPLES:
            rng = np.random.default_rng(random_state + fold)  # 每折不同种子，可复现
            normal_tr = rng.choice(normal_tr, size=IF_MAX_NORMAL_SAMPLES, replace=False)

        scaler = StandardScaler()
        X_normal_s = scaler.fit_transform(X[normal_tr])     # 只在 normal 上 fit μ、σ
        X_va_s = scaler.transform(X[va_idx])                # val 用同一 scaler transform（含欺诈）

        iforest = IsolationForest(
            n_estimators=IF_N_ESTIMATORS,
            max_samples=IF_MAX_SAMPLES,
            contamination=IF_CONTAMINATION,
            random_state=random_state + fold,
            n_jobs=-1,
        )
        iforest.fit(X_normal_s)
        # score_samples 越小越异常 → 取负后越大越异常，与 ae_oof_error 方向一致
        oof_score[va_idx] = -iforest.score_samples(X_va_s)
        print(f'  fold {fold}/{n_splits} 完成（normal 训练样本 {len(normal_tr):,}）')
    return oof_score


oof_if_anomaly_score._impl = OOF_IF_IMPL_VERSION
try:
    _fn_src = _inspect.getsource(oof_if_anomaly_score)
except OSError:
    _fn_src = ""
_dbg_log(
    "H5", "fe3:cell6", "cell6_loaded",
    {
        "version": OOF_IF_IMPL_VERSION,
        "fn_impl_attr": getattr(oof_if_anomaly_score, "_impl", "MISSING"),
        "fn_src_has_tf": "tf." in _fn_src,
        "fn_src_has_keras": "keras" in _fn_src,
        "fn_src_has_iforest": "IsolationForest" in _fn_src,
    },
)

# --- 执行：在 BASE 特征上跑一版 LGB，得到 Family A 要交叉的 Top-6 V ---
TOP_V = pick_top_v_features(df_model, BASE_FEATURES, k=TOP_V_K, model_name='LightGBM')


Top-6 V（LightGBM gain）：['V4', 'V14', 'V12', 'V16', 'V7', 'V26']


## 6.1 Family A — 1 EUR 特征交叉（FP 优先）

`one_euro_{Vi} = is_one_euro × V_i`，仅在 1 EUR 区激活对应主成分信号。



In [34]:
from IPython.display import display

# =============================================================================
# 6.1 Family A — 1 EUR 门控交叉 + ablation
# -----------------------------------------------------------------------------
# 流程：构造交叉列 → 对 LightGBM / XGBoost 分别做 feature_ablation
# ablation 基线默认为 BASE_FEATURES（V + Amount + Time），见 §1 的 feature_ablation
# =============================================================================

# build_cross_features 返回：
#   df_fe          — 在 df_model 上追加 one_euro_V* 列后的完整表
#   CROSS_FAMILY_A — 新列名列表，如 ['one_euro_V14', 'one_euro_V4', ...]
df_fe, CROSS_FAMILY_A = build_cross_features(
    df_model, TOP_V, gate_col='is_one_euro', prefix='one_euro',
)

print(f'Family A 共 {len(CROSS_FAMILY_A)} 列：')
# 展示「列名 → 定义」对照表，便于报告引用
display(pd.DataFrame({'特征': CROSS_FAMILY_A, '定义': [f'is_one_euro × {v}' for v in TOP_V]}))

# --- LightGBM ablation ---
# include_all_combo=True：除「基线」「+单列」外，再跑一行「+全部候选」（6 列一起加）
# 6.4 定稿用「+全部候选」行的 Δ AUC-PR 判断是否整族入选
print('\n=== Family A ablation（LightGBM，基线=BASE_FEATURES）===')
family_a_lgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='LightGBM', include_all_combo=True,
)
display(family_a_lgb.round(4))  # round(4) 便于阅读；Δ AUC-PR 相对基线行

# --- XGBoost ablation（同一划分、同一候选，对比两模型是否一致受益）---
print('\n=== Family A ablation（XGBoost）===')
family_a_xgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='XGBoost', include_all_combo=True,
)
display(family_a_xgb.round(4))


Family A 共 6 列：


,特征,定义
0,one_euro_V4,is_one_euro × V4
1,one_euro_V14,is_one_euro × V14
2,one_euro_V12,is_one_euro × V12
3,one_euro_V16,is_one_euro × V16
4,one_euro_V7,is_one_euro × V7
5,one_euro_V26,is_one_euro × V26



=== Family A ablation（LightGBM，基线=BASE_FEATURES）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+one_euro_V14,31,0.8815,0.8660,0.8750,0.8571,12,14,0.0049
1,+全部候选,36,0.8786,0.8601,0.8737,0.8469,12,15,0.0020
2,+one_euro_V16,31,0.8781,0.8601,0.8737,0.8469,12,15,0.0015
3,+one_euro_V12,31,0.8773,0.8601,0.8737,0.8469,12,15,0.0007
4,基线,30,0.8766,0.8557,0.8646,0.8469,13,15,0.0000
5,+one_euro_V7,31,0.8731,0.8601,0.8737,0.8469,12,15,-0.0035
6,+one_euro_V26,31,0.8727,0.8557,0.8646,0.8469,13,15,-0.0039
7,+one_euro_V4,31,0.8722,0.8542,0.8723,0.8367,12,16,-0.0044



=== Family A ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+one_euro_V26,31,0.8750,0.8542,0.8723,0.8367,12,16,0.0049
1,+one_euro_V14,31,0.8729,0.8526,0.8804,0.8265,11,17,0.0028
2,+one_euro_V12,31,0.8718,0.8526,0.8804,0.8265,11,17,0.0017
3,基线,30,0.8701,0.8571,0.8901,0.8265,10,17,0.0000
4,+one_euro_V4,31,0.8697,0.8482,0.8710,0.8265,12,17,-0.0004
5,+one_euro_V16,31,0.8691,0.8556,0.8989,0.8163,9,18,-0.0010
6,+全部候选,36,0.8671,0.8377,0.8602,0.8163,13,18,-0.0030
7,+one_euro_V7,31,0.8617,0.8466,0.8791,0.8163,11,18,-0.0084


## 6.2 Family B（占位）+ Family C 说明

**Family B（待做）：** `band_{Vi} = is_amount_1_30 × V_i` 或 `is_amount_75_110 × V_i`，针对难样本金额带内 FN。
镜像 6.1，将 `gate_col='is_amount_1_30'`（或 `is_amount_75_110`）、`prefix='band_low'` 即可。

**Family C（本轮不做）：** 可选 `log1p_amount × is_amount_1_30`、`log1p_amount × V_top` 等可解释连续门控。



In [35]:
# =============================================================================
# 6.2 Family B — 占位（尚未实现）
# -----------------------------------------------------------------------------
# 设计：与 6.1 镜像，门控改为 is_amount_1_30 / is_amount_75_110，针对难样本金额带 FN。
# 取消注释下方代码即可跑通；TOP_V 可复用 6.0 结果，也可在 BASE 上重新 pick_top_v。
# =============================================================================

# TODO: 待做 — 镜像 6.1
# df_fe, CROSS_FAMILY_B = build_cross_features(
#     df_fe, TOP_V, gate_col='is_amount_1_30', prefix='band_low',
# )
# family_b_lgb = feature_ablation(
#     df_fe, candidate_features=CROSS_FAMILY_B,
#     model_name='LightGBM', include_all_combo=True,
# )

print('Family B 尚未实现；见上方 markdown 说明。')


Family B 尚未实现；见上方 markdown 说明。


## 6.3 OOF Isolation Forest 重构误差

每折 **仅用正常样本** 训练 Isolation Forest；折外异常分作为 `if_oof_score`（越大越异常（-score_samples））。
与树模型 **不同数学原理**，非树模型 `p_fraud`，无目标泄露。



In [36]:
from IPython.display import display

# =============================================================================
# 6.3 OOF Isolation Forest — 计算 if_oof_score 并 ablation
# -----------------------------------------------------------------------------
# if_oof_score：与树模型不同原理的异常分，可作为 BASE 上的增量特征。
# 注意：全表 5-fold OOF 较慢（每折训一个 IF），首次运行需数分钟。
# =============================================================================

# #region agent log
import inspect as _inspect
if "_dbg_log" not in globals():
    def _dbg_log(hypothesis_id, location, message, data=None, run_id="pre-fix"):
        import json as _json, time as _time
        payload = {
            "sessionId": "db9af8",
            "runId": run_id,
            "hypothesisId": hypothesis_id,
            "location": location,
            "message": message,
            "data": data or {},
            "timestamp": int(_time.time() * 1000),
        }
        with open("/Users/jingyuhe/.cursor/debug-logs/debug-db9af8.log", "a", encoding="utf-8") as _f:
            _f.write(_json.dumps(payload, ensure_ascii=False) + "\n")

try:
    _src = _inspect.getsource(oof_if_anomaly_score)
except OSError:
    _src = ""
_impl = getattr(oof_if_anomaly_score, "_impl", "MISSING")
_stale = (_impl != "isolation_forest_v1") or ("tf." in _src) or ("keras" in _src) or ("_build_ae" in _src)
_version = globals().get("OOF_IF_IMPL_VERSION", "MISSING")
_dbg_log(
    "H1", "fe3:cell12", "before_oof_call",
    {
        "impl_version": _version,
        "fn_impl_attr": _impl,
        "fn_src_has_tf": "tf." in _src,
        "fn_src_has_keras": "keras" in _src,
        "fn_src_has_iforest": "IsolationForest" in _src,
        "stale_detected": _stale,
        "oof_fn_defined": "oof_if_anomaly_score" in globals(),
    },
)
if _stale or _version != "isolation_forest_v1":
    raise RuntimeError(
        "检测到内存中仍是旧版 AE 版 oof_if_anomaly_score（含 tf/keras）。"
        "请 Kernel → Restart，然后从 §0 依次运行到 §6.3；或至少重新运行 §6.0 后再运行 §6.3。"
    )
# #endregion

print('计算 OOF IF 重构误差（5-fold，可能需数分钟）…')
# 返回值长度 = len(df_fe)，按 index 对齐写入新列
df_fe['if_oof_score'] = oof_if_anomaly_score(df_fe, feature_cols=IF_INPUT_COLS)
print(f"if_oof_score: mean={df_fe['if_oof_score'].mean():.6f}, "
      f"std={df_fe['if_oof_score'].std():.6f}")

# --- 单特征 ablation：只加 if_oof_score 一列 ---
# include_all_combo=False：候选只有 1 列时无需「+全部候选」行
print('\n=== OOF IF ablation（LightGBM）===')
if_ablation_lgb = feature_ablation(
    df_fe, candidate_features=['if_oof_score'],
    model_name='LightGBM', include_all_combo=False,
)
display(if_ablation_lgb.round(4))

print('\n=== OOF IF ablation（XGBoost）===')
if_ablation_xgb = feature_ablation(
    df_fe, candidate_features=['if_oof_score'],
    model_name='XGBoost', include_all_combo=False,
)
display(if_ablation_xgb.round(4))

# --- 可选：AE 误差 × 1 EUR 门控 ---
# 假设：异常分在 1 EUR 子空间更有判别力；乘 is_one_euro 后非 1 EUR 行为 0
# base_features 显式设为 BASE + if_oof_score，测「在已有 IF 分上再加门控交叉」的边际收益
df_fe['if_oof_score_x_one_euro'] = df_fe['if_oof_score'] * df_fe['is_one_euro'].astype(float)
print('\n=== if_oof_score × is_one_euro ablation（LightGBM）===')
ae_gate_ablation = feature_ablation(
    df_fe, base_features=BASE_FEATURES + ['if_oof_score'],
    candidate_features=['if_oof_score_x_one_euro'],
    model_name='LightGBM', include_all_combo=False,
)
display(ae_gate_ablation.round(4))


计算 OOF IF 重构误差（5-fold，可能需数分钟）…
  fold 1/5 完成（normal 训练样本 50,000）
  fold 2/5 完成（normal 训练样本 50,000）
  fold 3/5 完成（normal 训练样本 50,000）
  fold 4/5 完成（normal 训练样本 50,000）
  fold 5/5 完成（normal 训练样本 50,000）
if_oof_score: mean=0.343322, std=0.027824

=== OOF IF ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8766,0.8557,0.8646,0.8469,13,15,0.0000
1,+if_oof_score,31,0.8722,0.8513,0.8557,0.8469,14,15,-0.0044



=== OOF IF ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+if_oof_score,31,0.8748,0.8632,0.8913,0.8367,10,16,0.0047
1,基线,30,0.8701,0.8571,0.8901,0.8265,10,17,0.0000



=== if_oof_score × is_one_euro ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+if_oof_score_x_one_euro,32,0.8806,0.8542,0.8723,0.8367,12,16,0.0084
1,基线,31,0.8722,0.8513,0.8557,0.8469,14,15,0.0000


## 6.4 组合验证 + 定稿 MODEL_FEATURES_V2

**6.4a**（5-fold CV，LGB / XGB **分表**）覆盖 **全部人为特征** 及组合：

| 块 | 内容 |
|----|------|
| **EDA** | `log1p_amount`、`hours_since_start`、`is_micro_testing`、`is_one_euro`、`is_amount_1_30`（(1,30]）、`is_amount_75_110`（[75,110]）；无 `is_inactive` / `is_small_testing` |
| **Family A** | `one_euro×Top-V`（单列 / Top-2/3 / 全族） |
| **OOF IF** | `if_oof_score`、`if_oof_score×is_one_euro` |
| **跨块** | EDA+AE、EDA+Family A、EDA+IF+Family A、**全部人为特征** |

**6.4b**：双模型 CV Δ 均为正且 Δ_mean 最高 → 定稿 `MODEL_FEATURES_V2`。



In [37]:
from IPython.display import display

# =============================================================================
# 6.4a 全部人为特征组合矩阵（EDA + Family A + OOF IF；5-fold CV；LGB/XGB 分表）
# =============================================================================

FE_EDA = list(EDA_FEATURES)  # log1p, hours, micro, one_euro, 难样本金额带 (1,30]/[75,110]
FE_IF = ['if_oof_score']
FE_IF_GATE = ['if_oof_score_x_one_euro']
FE_FAMILY_A = list(CROSS_FAMILY_A)
EDA_CURATED = ['hours_since_start', 'is_one_euro']  # 主 notebook 2.1d curated


def _dedupe_preserve_order(cols: list) -> list:
    seen, out = set(), []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def _extra_summary(extra_cols: list) -> str:
    if not extra_cols:
        return '—'
    s = ', '.join(extra_cols)
    return s if len(s) <= 72 else s[:69] + '...'


def _dedupe_specs(specs: list[tuple[list, str]]) -> list[tuple[list, str]]:
    seen: set[tuple[str, ...]] = set()
    out: list[tuple[list, str]] = []
    for extra, label in specs:
        key = tuple(_dedupe_preserve_order(extra))
        if key in seen:
            continue
        seen.add(key)
        out.append((list(key), label))
    return out


def build_fe_combo_specs(
    family_a_cols: list,
    eda_cols: list | None = None,
    fe_if: list | None = None,
    fe_if_gate: list | None = None,
    top_k_subsets: tuple[int, ...] = (2, 3),
) -> list[tuple[list, str]]:
    """
    构造 **全部人为特征** 的组合规格（相对 BASE_FEATURES 的增量列）。
    块：EDA 1.7 → OOF IF → Family A → 跨块并集。
    """
    eda_cols = list(eda_cols or FE_EDA)
    fe_if = list(fe_if or FE_IF)
    fe_if_gate = list(fe_if_gate or FE_IF_GATE)
    specs: list[tuple[list, str]] = []

    eda_minus_bands = [c for c in eda_cols if c not in AMOUNT_BAND_FEATURES]
    eda_minus_no_hours = [c for c in eda_minus_bands if c != 'hours_since_start']
    eda_minus_no_hours_log = [c for c in eda_minus_no_hours if c != 'log1p_amount']

    # --- 0. 基线 ---
    specs.append(([], '0. 基线（仅 BASE）'))

    # --- EDA：单列 ---
    for i, col in enumerate(eda_cols, start=1):
        specs.append(([col], f'Ed{i}. BASE + {col}'))

    # --- EDA：2.1d 式组合（镜像主 notebook）---
    specs.append((eda_cols, 'Ed_all. BASE + 全部 EDA（6 列）'))
    specs.append((eda_minus_bands, 'Ed_noBands. BASE + EDA 去掉难样本金额带'))
    specs.append((list(AMOUNT_BAND_FEATURES), 'Ed_bands. BASE + 两档难样本金额带'))
    specs.append((eda_minus_no_hours, 'Ed_noHours. BASE + EDA 再去掉 hours'))
    specs.append((eda_minus_no_hours_log, 'Ed_noHoursLog. BASE + EDA 再去掉 log1p'))
    specs.append((EDA_CURATED, 'Ed_cur. BASE + hours + is_one_euro'))

    # --- OOF IF ---
    specs.append((fe_if, 'IF. BASE + if_oof_score'))
    specs.append((fe_if + fe_if_gate, 'IF+gate. BASE + if_oof_score + if×1EUR'))

    # --- EDA × AE ---
    specs.append((eda_cols + fe_if, 'Ed_all+IF. 全部 EDA + if_oof_score'))
    specs.append((EDA_CURATED + fe_if, 'Ed_cur+IF. hours+is_one_euro + if_oof_score'))
    specs.append((eda_minus_no_hours_log + fe_if, 'Ed_noHoursLog+IF. EDA(无金额带/hours/log1p) + AE'))
    for col in eda_cols:
        specs.append((fe_if + [col], f'IF+{col}. if_oof_score + {col}'))

    # --- Family A：单列 / +IF / Top-k / 全族 ---
    for i, col in enumerate(family_a_cols, start=1):
        short = col.replace('one_euro_', '')
        specs.append(([col], f'A{i}. BASE + {col}（{short}）'))
        specs.append((fe_if + [col], f'A{i}+IF. BASE + if + {col}（{short}）'))
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        v_short = '+'.join(c.replace('one_euro_', '') for c in subset)
        specs.append((subset, f'A_top{k}. Family A Top-{k}（{v_short}）'))
        specs.append((fe_if + subset, f'A_top{k}+IF. ae + Family A Top-{k}'))
    specs.append((family_a_cols, 'A_all. Family A 全族（6 列）'))
    specs.append((fe_if + family_a_cols, 'A_all+IF. ae + Family A 全族'))

    # --- EDA × Family A（Top-k，不加 AE）---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + subset, f'Ed_all+A_top{k}. 全部 EDA + Family A Top-{k}'))
        specs.append((EDA_CURATED + subset, f'Ed_cur+A_top{k}. curated EDA + Family A Top-{k}'))

    # --- EDA × AE × Family A ---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + fe_if + subset, f'Ed_all+IF+A_top{k}. 全部 EDA + ae + Family A Top-{k}'))
        specs.append((EDA_CURATED + fe_if + subset, f'Ed_cur+IF+A_top{k}. curated + ae + Family A Top-{k}'))

    # --- 全量人为特征（EDA + AE + Family A + AE 门控）---
    all_handcrafted = _dedupe_preserve_order(eda_cols + fe_if + family_a_cols + fe_if_gate)
    specs.append((all_handcrafted, 'FULL. BASE + 全部人为特征（EDA+IF+FamilyA+gate）'))

    return _dedupe_specs(specs)


def eval_fe_combo(
    data: pd.DataFrame,
    extra_cols: list,
    label: str,
    model_name: str = 'LightGBM',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """5-fold CV 评估 BASE + extra_cols（比单次 hold-out 更稳）。"""
    extra_cols = _dedupe_preserve_order(list(extra_cols))
    feature_cols = _dedupe_preserve_order(BASE_FEATURES + extra_cols)
    missing = [c for c in feature_cols if c not in data.columns]
    if missing:
        raise KeyError(f'[{label}] 缺少列: {missing}')

    res = cross_val_eval(
        model_name, data, feature_cols,
        n_splits=n_splits, random_state=random_state, threshold=threshold,
    )
    return {
        '特征组合': label,
        '增量列数': len(extra_cols),
        '总特征数': len(feature_cols),
        '增量摘要': _extra_summary(extra_cols),
        '增量列': extra_cols,
        'AUC-PR_mean': res['AUC-PR_mean'],
        'AUC-PR_std': res['AUC-PR_std'],
        f'F1@{threshold}': res[f'F1@{threshold}'],
        f'Precision@{threshold}': res[f'Precision@{threshold}'],
        f'Recall@{threshold}': res[f'Recall@{threshold}'],
        'FP': res['FP'],
        'FN': res['FN'],
    }


def run_fe_combo_matrix(
    data: pd.DataFrame,
    combo_specs: list[tuple[list, str]],
    model_name: str = 'LightGBM',
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []
    n = len(combo_specs)
    for i, (extra, label) in enumerate(combo_specs, start=1):
        if verbose:
            print(f'  [{model_name}] CV {i}/{n}: {label}')
        rows.append(eval_fe_combo(data, extra, label, model_name=model_name))
    df_out = pd.DataFrame(rows)
    base_ap = float(df_out.loc[df_out['特征组合'].str.startswith('0.'), 'AUC-PR_mean'].iloc[0])
    df_out['Δ AUC-PR vs BASE'] = df_out['AUC-PR_mean'] - base_ap
    return df_out.sort_values('AUC-PR_mean', ascending=False).reset_index(drop=True)


def _display_combo_table(df: pd.DataFrame, title: str) -> None:
    """展示组合表（隐藏增量列 list，用增量摘要代替）。"""
    show_cols = [c for c in df.columns if c != '增量列']
    print(title)
    display(df[show_cols].round(4))


COMBO_SPECS = build_fe_combo_specs(FE_FAMILY_A)
_n_eda = len(FE_EDA)
print(f'共 {len(COMBO_SPECS)} 种组合 × {CV_N_SPLITS}-fold CV')
print(f'  含 EDA 1.7（{_n_eda} 列单列+组合）、Family A、OOF IF、跨块 FULL 等\n')

print('=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_lgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='LightGBM')
_display_combo_table(combo_lgb, '')

print(f'\n=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_xgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='XGBoost')
_display_combo_table(combo_xgb, '')

print('\n--- LightGBM：CV Δ>0 的前 3 名 ---')
display(combo_lgb.loc[combo_lgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

print('--- XGBoost：CV Δ>0 的前 3 名 ---')
display(combo_xgb.loc[combo_xgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

# 供 6.4b 定稿：内部合并，不在此 cell 展示宽表
combo_lgb_idx = combo_lgb.set_index('特征组合')
combo_xgb_idx = combo_xgb.set_index('特征组合')
_common_labels = combo_lgb_idx.index.intersection(combo_xgb_idx.index)
combo_dual_rows = []
for label in _common_labels:
    dl = float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    dx = float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    combo_dual_rows.append({
        '特征组合': label,
        '增量列': combo_lgb_idx.loc[label, '增量列'],
        'Δ_LGB': dl,
        'Δ_XGB': dx,
        'Δ_mean': (dl + dx) / 2,
        '双模型Δ均为正': dl > 0 and dx > 0,
    })
combo_dual = pd.DataFrame(combo_dual_rows).sort_values('Δ_mean', ascending=False).reset_index(drop=True)


共 51 种组合 × 5-fold CV
  含 EDA 1.7（6 列单列+组合）、Family A、OOF IF、跨块 FULL 等

=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [LightGBM] CV 1/51: 0. 基线（仅 BASE）
  [LightGBM] CV 2/51: Ed1. BASE + log1p_amount
  [LightGBM] CV 3/51: Ed2. BASE + hours_since_start
  [LightGBM] CV 4/51: Ed3. BASE + is_micro_testing
  [LightGBM] CV 5/51: Ed4. BASE + is_one_euro
  [LightGBM] CV 6/51: Ed5. BASE + is_amount_1_30
  [LightGBM] CV 7/51: Ed6. BASE + is_amount_75_110
  [LightGBM] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [LightGBM] CV 9/51: Ed_noBands. BASE + EDA 去掉难样本金额带
  [LightGBM] CV 10/51: Ed_bands. BASE + 两档难样本金额带
  [LightGBM] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [LightGBM] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [LightGBM] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [LightGBM] CV 14/51: IF. BASE + if_oof_score
  [LightGBM] CV 15/51: IF+gate. BASE + if_oof_score + if×1EUR
  [LightGBM] CV 16/51: Ed_all+IF. 全部 EDA + if_oof_score
  [LightGBM] CV 17/51: Ed_cur+IF. hours+is_one_euro + if_oof_

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_all+A_top2. 全部 EDA + Family A Top-2,8,38,"log1p_amount, hours_since_start, is_micro_test...",0.8602,0.0253,0.8553,0.8901,0.8232,50,87,0.0030
1,Ed_noHours. BASE + EDA 再去掉 hours,3,33,"log1p_amount, is_micro_testing, is_one_euro",0.8601,0.0234,0.8581,0.8960,0.8232,47,87,0.0028
2,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8600,0.0271,0.8605,0.8965,0.8272,47,85,0.0028
3,Ed_bands. BASE + 两档难样本金额带,2,32,"is_amount_1_30, is_amount_75_110",0.8599,0.0268,0.8602,0.8982,0.8252,46,86,0.0027
4,Ed_cur+IF. hours+is_one_euro + if_oof_score,3,33,"hours_since_start, is_one_euro, if_oof_score",0.8599,0.0264,0.8586,0.8925,0.8272,49,85,0.0027
5,Ed_cur+IF+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, if_oof_score, ...",0.8594,0.0268,0.8571,0.8940,0.8232,48,87,0.0022
6,Ed2. BASE + hours_since_start,1,31,hours_since_start,0.8591,0.0251,0.8571,0.8940,0.8232,48,87,0.0019
7,IF. BASE + if_oof_score,1,31,if_oof_score,0.8591,0.0263,0.8584,0.8943,0.8252,48,86,0.0018
8,A3+IF. BASE + if + one_euro_V12（V12）,2,32,"if_oof_score, one_euro_V12",0.8591,0.0263,0.8596,0.8945,0.8272,48,85,0.0018
9,A6+IF. BASE + if + one_euro_V26（V26）,2,32,"if_oof_score, one_euro_V26",0.8590,0.0249,0.8535,0.8862,0.8232,52,87,0.0018



=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [XGBoost] CV 1/51: 0. 基线（仅 BASE）
  [XGBoost] CV 2/51: Ed1. BASE + log1p_amount
  [XGBoost] CV 3/51: Ed2. BASE + hours_since_start
  [XGBoost] CV 4/51: Ed3. BASE + is_micro_testing
  [XGBoost] CV 5/51: Ed4. BASE + is_one_euro
  [XGBoost] CV 6/51: Ed5. BASE + is_amount_1_30
  [XGBoost] CV 7/51: Ed6. BASE + is_amount_75_110
  [XGBoost] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [XGBoost] CV 9/51: Ed_noBands. BASE + EDA 去掉难样本金额带
  [XGBoost] CV 10/51: Ed_bands. BASE + 两档难样本金额带
  [XGBoost] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [XGBoost] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [XGBoost] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [XGBoost] CV 14/51: IF. BASE + if_oof_score
  [XGBoost] CV 15/51: IF+gate. BASE + if_oof_score + if×1EUR
  [XGBoost] CV 16/51: Ed_all+IF. 全部 EDA + if_oof_score
  [XGBoost] CV 17/51: Ed_cur+IF. hours+is_one_euro + if_oof_score
  [XGBoost] CV 18/51: Ed_noHoursLog+IF. EDA(无金额带/hours/log1p) + AE
  [XGBoost] CV

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A_top3. Family A Top-3（V4+V14+V12）,3,33,"one_euro_V4, one_euro_V14, one_euro_V12",0.8600,0.0284,0.8574,0.8923,0.8252,49,86,0.0030
1,A_top2. Family A Top-2（V4+V14）,2,32,"one_euro_V4, one_euro_V14",0.8588,0.0273,0.8545,0.8812,0.8293,55,84,0.0019
2,A5. BASE + one_euro_V7（V7）,1,31,one_euro_V7,0.8587,0.0286,0.8550,0.8848,0.8272,53,85,0.0018
3,Ed_cur+IF+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, if_oof_score, ...",0.8583,0.0294,0.8520,0.8807,0.8252,55,86,0.0014
4,Ed_all. BASE + 全部 EDA（6 列）,6,36,"log1p_amount, hours_since_start, is_micro_test...",0.8582,0.0300,0.8571,0.8870,0.8293,52,84,0.0013
5,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8581,0.0286,0.8577,0.8906,0.8272,50,85,0.0011
6,A2. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.8574,0.0289,0.8598,0.8858,0.8354,53,81,0.0005
7,Ed_cur+A_top3. curated EDA + Family A Top-3,5,35,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8573,0.0287,0.8544,0.8882,0.8232,51,87,0.0004
8,A6. BASE + one_euro_V26（V26）,1,31,one_euro_V26,0.8573,0.0300,0.8574,0.8853,0.8313,53,83,0.0004
9,Ed_all+IF+A_top2. 全部 EDA + ae + Family A Top-2,9,39,"log1p_amount, hours_since_start, is_micro_test...",0.8571,0.0279,0.8541,0.8829,0.8272,54,85,0.0002



--- LightGBM：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_all+A_top2. 全部 EDA + Family A Top-2,8,38,"log1p_amount, hours_since_start, is_micro_test...",0.8602,0.0253,0.8553,0.8901,0.8232,50,87,0.0030
1,Ed_noHours. BASE + EDA 再去掉 hours,3,33,"log1p_amount, is_micro_testing, is_one_euro",0.8601,0.0234,0.8581,0.8960,0.8232,47,87,0.0028
2,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8600,0.0271,0.8605,0.8965,0.8272,47,85,0.0028


--- XGBoost：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A_top3. Family A Top-3（V4+V14+V12）,3,33,"one_euro_V4, one_euro_V14, one_euro_V12",0.8600,0.0284,0.8574,0.8923,0.8252,49,86,0.0030
1,A_top2. Family A Top-2（V4+V14）,2,32,"one_euro_V4, one_euro_V14",0.8588,0.0273,0.8545,0.8812,0.8293,55,84,0.0019
2,A5. BASE + one_euro_V7（V7）,1,31,one_euro_V7,0.8587,0.0286,0.8550,0.8848,0.8272,53,85,0.0018


定稿后可在 `Ed_bands`（两档金额带）、`Ed_cur+A_top3`、`Ed_noBands` 等候选间，用 **1 EUR 子集 FP** 做最终业务验收。


In [38]:
# =============================================================================
# 6.4b 定稿 MODEL_FEATURES_V2
# -----------------------------------------------------------------------------
# 展示：LGB / XGB 各自最优（Δ>0）；定稿：双模型 Δ 均为正且 Δ_mean 最高（combo_dual）
# =============================================================================

def _ablation_combo_delta(ablation_df: pd.DataFrame) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == '+全部候选']
    if row.empty:
        row = ablation_df.iloc[[0]]
    return float(row['Δ AUC-PR'].iloc[0])


def _ablation_single_delta(ablation_df: pd.DataFrame, tag: str) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == tag]
    return float(row['Δ AUC-PR'].iloc[0]) if not row.empty else 0.0


def _best_positive_row(combo_df: pd.DataFrame) -> pd.Series | None:
    pos = combo_df.loc[combo_df['Δ AUC-PR vs BASE'] > 0]
    return pos.iloc[0] if not pos.empty else None


decisions = []

# --- 各模型单独最优（仅展示，不一定作最终定稿）---
best_lgb = _best_positive_row(combo_lgb)
best_xgb = _best_positive_row(combo_xgb)

print('=== 6.4b LightGBM 单模型最优（CV Δ>0）===')
if best_lgb is not None:
    print(f"  {best_lgb['特征组合']}  |  Δ={best_lgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_lgb['AUC-PR_mean']:.4f}±{best_lgb['AUC-PR_std']:.4f}  |  {best_lgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

print('\n=== 6.4b XGBoost 单模型最优（CV Δ>0）===')
if best_xgb is not None:
    print(f"  {best_xgb['特征组合']}  |  Δ={best_xgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_xgb['AUC-PR_mean']:.4f}±{best_xgb['AUC-PR_std']:.4f}  |  {best_xgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

# --- 双模型一致定稿 ---
eligible = combo_dual[combo_dual['双模型Δ均为正']].sort_values('Δ_mean', ascending=False)
if eligible.empty:
    WINNER_LABEL = '0. 基线（仅 BASE）'
    SELECTED_EXTRA = []
    decisions.append('组合定稿：无「双模型 CV Δ 均为正」方案 → 保留 BASE_FEATURES')
else:
    winner = eligible.iloc[0]
    WINNER_LABEL = winner['特征组合']
    SELECTED_EXTRA = list(winner['增量列'])
    decisions.append(
        f"组合定稿：{WINNER_LABEL} "
        f"(LGB Δ={winner['Δ_LGB']:.4f}, XGB Δ={winner['Δ_XGB']:.4f}, Δ_mean={winner['Δ_mean']:.4f})"
    )

MODEL_FEATURES_V2 = BASE_FEATURES + [c for c in SELECTED_EXTRA if c not in BASE_FEATURES]

decisions.append(
    f'[参考] Family A 单族 ablation: LGB Δ={_ablation_combo_delta(family_a_lgb):.4f}, '
    f'XGB Δ={_ablation_combo_delta(family_a_xgb):.4f}'
)
decisions.append(
    f'[参考] if_oof_score 单列 ablation: LGB Δ={_ablation_single_delta(if_ablation_lgb, "+if_oof_score"):.4f}, '
    f'XGB Δ={_ablation_single_delta(if_ablation_xgb, "+if_oof_score"):.4f}'
)

print('\n=== 6.4b 定稿决策 ===')
for d in decisions:
    print(' -', d)
print(f'\nMODEL_FEATURES_V2（{len(MODEL_FEATURES_V2)} 列，相对 BASE 增量 {len(MODEL_FEATURES_V2) - len(BASE_FEATURES)}）：')
print(MODEL_FEATURES_V2)

print('\n提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。')

import json as _json
export_path = 'MODEL_FEATURES_V2_hardband.json'
with open(export_path, 'w', encoding='utf-8') as f:
    _json.dump(
        {
            'MODEL_FEATURES_V2': MODEL_FEATURES_V2,
            'winner_combo': WINNER_LABEL,
            'selected_extra': SELECTED_EXTRA,
            'best_lgb_combo': None if best_lgb is None else best_lgb['特征组合'],
            'best_xgb_combo': None if best_xgb is None else best_xgb['特征组合'],
            'cv_n_splits': CV_N_SPLITS,
            'es_frac': ES_FRAC,
            'eda_features': EDA_FEATURES,
            'eda_feature_doc': EDA_FEATURE_DOC,
            'decisions': decisions,
            'combo_lgb': combo_lgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
            'combo_xgb': combo_xgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
        },
        f, ensure_ascii=False, indent=2,
    )
print(f'已导出 → {export_path}')


=== 6.4b LightGBM 单模型最优（CV Δ>0）===
  Ed_all+A_top2. 全部 EDA + Family A Top-2  |  Δ=0.0030  |  AUC-PR=0.8602±0.0253  |  log1p_amount, hours_since_start, is_micro_testing, is_one_euro, is_am...

=== 6.4b XGBoost 单模型最优（CV Δ>0）===
  A_top3. Family A Top-3（V4+V14+V12）  |  Δ=0.0030  |  AUC-PR=0.8600±0.0284  |  one_euro_V4, one_euro_V14, one_euro_V12

=== 6.4b 定稿决策 ===
 - 组合定稿：A_top3. Family A Top-3（V4+V14+V12） (LGB Δ=0.0010, XGB Δ=0.0030, Δ_mean=0.0020)
 - [参考] Family A 单族 ablation: LGB Δ=0.0020, XGB Δ=-0.0030
 - [参考] if_oof_score 单列 ablation: LGB Δ=-0.0044, XGB Δ=0.0047

MODEL_FEATURES_V2（33 列，相对 BASE 增量 3）：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time', 'one_euro_V4', 'one_euro_V14', 'one_euro_V12']

提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。
已导出 → MODEL_FEATURES_V2_hardband.json


## 6.4c 补测：难样本金额带 + Family A Top-k（轻量）

**前置：** 已跑过 §1（`eval_fe_combo`）、§6.0–6.1（`df_fe`、`CROSS_FAMILY_A`）。**不必**重跑 6.4a 全矩阵。

只 CV 评估此前矩阵**未单独列出**的组合：`Ed_bands + A_top2/3`，以及可选的 `Ed_bands + AE + A_top3`。


In [39]:
# =============================================================================
# 6.4c 补测：Ed_bands + A_top（+ 可选 AE）— 5-fold CV，LGB / XGB 各跑一遍
# =============================================================================
from IPython.display import display

_required = ('df_fe', 'eval_fe_combo', 'AMOUNT_BAND_FEATURES', 'CROSS_FAMILY_A')
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(f'请先运行 §1 与 §6.0–6.1，缺少: {_missing}')

_family_a = list(CROSS_FAMILY_A)
_bands = list(AMOUNT_BAND_FEATURES)

QUICK_BAND_ATOP_SPECS = [
    (_bands, 'Ed_bands. BASE + 两档难样本金额带'),
    (_family_a[:2], 'A_top2. Family A Top-2'),
    (_family_a[:3], 'A_top3. Family A Top-3'),
    (_bands + _family_a[:2], 'Ed_bands+A_top2. 金额带 + Family A Top-2'),
    (_bands + _family_a[:3], 'Ed_bands+A_top3. 金额带 + Family A Top-3'),
]
if 'if_oof_score' in df_fe.columns:
    QUICK_BAND_ATOP_SPECS.append(
        (_bands + ['if_oof_score'] + _family_a[:3],
         'Ed_bands+IF+A_top3. 金额带 + ae + Family A Top-3')
    )

print(f'补测 {len(QUICK_BAND_ATOP_SPECS)} 种组合 × {CV_N_SPLITS}-fold CV\n')


def _base_ap_from_prior(model_name: str) -> float | None:
    tbl = 'combo_lgb' if model_name == 'LightGBM' else 'combo_xgb'
    if tbl not in globals():
        return None
    ref = globals()[tbl]
    row = ref.loc[ref['特征组合'].str.startswith('0.'), 'AUC-PR_mean']
    return float(row.iloc[0]) if len(row) else None


def _run_quick_matrix(data, specs, model_name: str) -> pd.DataFrame:
    rows = []
    for extra, label in specs:
        print(f'  [{model_name}] {label}')
        rows.append(eval_fe_combo(data, extra, label, model_name=model_name))
    out = pd.DataFrame(rows)
    base_ap = _base_ap_from_prior(model_name)
    if base_ap is None:
        print(f'  [{model_name}] （无 6.4a 结果）补跑 BASE…')
        base_ap = float(eval_fe_combo(data, [], '0. 基线（仅 BASE）', model_name=model_name)['AUC-PR_mean'])
    out['Δ AUC-PR vs BASE'] = out['AUC-PR_mean'] - base_ap
    return out.sort_values('AUC-PR_mean', ascending=False).reset_index(drop=True)


print('=== 6.4c LightGBM ===')
quick_lgb = _run_quick_matrix(df_fe, QUICK_BAND_ATOP_SPECS, 'LightGBM')
display(quick_lgb.drop(columns=['增量列'], errors='ignore').round(4))

print('\n=== 6.4c XGBoost ===')
quick_xgb = _run_quick_matrix(df_fe, QUICK_BAND_ATOP_SPECS, 'XGBoost')
display(quick_xgb.drop(columns=['增量列'], errors='ignore').round(4))

# 与 6.4a 已有行对照（若存在）
for tbl_name, quick_df in [('combo_lgb', quick_lgb), ('combo_xgb', quick_xgb)]:
    if tbl_name not in globals():
        continue
    ref = globals()[tbl_name]
    overlap = ref[ref['特征组合'].isin(quick_df['特征组合'])]
    if not overlap.empty:
        tag = 'LightGBM' if tbl_name == 'combo_lgb' else 'XGBoost'
        print(f'\n--- 与 6.4a {tag} 对照（同名组合 AP 应一致）---')
        display(overlap[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']].round(4))


补测 6 种组合 × 5-fold CV

=== 6.4c LightGBM ===
  [LightGBM] Ed_bands. BASE + 两档难样本金额带
  [LightGBM] A_top2. Family A Top-2
  [LightGBM] A_top3. Family A Top-3
  [LightGBM] Ed_bands+A_top2. 金额带 + Family A Top-2
  [LightGBM] Ed_bands+A_top3. 金额带 + Family A Top-3
  [LightGBM] Ed_bands+IF+A_top3. 金额带 + ae + Family A Top-3


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_bands. BASE + 两档难样本金额带,2,32,"is_amount_1_30, is_amount_75_110",0.8599,0.0268,0.8602,0.8982,0.8252,46,86,0.0027
1,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8588,0.0273,0.8617,0.8967,0.8293,47,84,0.0015
2,A_top3. Family A Top-3,3,33,"one_euro_V4, one_euro_V14, one_euro_V12",0.8583,0.0261,0.8608,0.9020,0.8232,44,87,0.0010
3,A_top2. Family A Top-2,2,32,"one_euro_V4, one_euro_V14",0.8581,0.0272,0.8577,0.8906,0.8272,50,85,0.0008
4,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8557,0.0271,0.8581,0.8960,0.8232,47,87,-0.0015
5,Ed_bands+IF+A_top3. 金额带 + ae + Family A Top-3,6,36,"is_amount_1_30, is_amount_75_110, if_oof_score...",0.8543,0.0274,0.8538,0.8916,0.8191,49,89,-0.0029



=== 6.4c XGBoost ===
  [XGBoost] Ed_bands. BASE + 两档难样本金额带
  [XGBoost] A_top2. Family A Top-2
  [XGBoost] A_top3. Family A Top-3
  [XGBoost] Ed_bands+A_top2. 金额带 + Family A Top-2
  [XGBoost] Ed_bands+A_top3. 金额带 + Family A Top-3
  [XGBoost] Ed_bands+IF+A_top3. 金额带 + ae + Family A Top-3


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8603,0.0275,0.8577,0.8906,0.8272,50,85,0.0034
1,A_top3. Family A Top-3,3,33,"one_euro_V4, one_euro_V14, one_euro_V12",0.8600,0.0284,0.8574,0.8923,0.8252,49,86,0.0030
2,A_top2. Family A Top-2,2,32,"one_euro_V4, one_euro_V14",0.8588,0.0273,0.8545,0.8812,0.8293,55,84,0.0019
3,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8579,0.0274,0.8599,0.8928,0.8293,49,84,0.0010
4,Ed_bands. BASE + 两档难样本金额带,2,32,"is_amount_1_30, is_amount_75_110",0.8563,0.0285,0.8541,0.8829,0.8272,54,85,-0.0006
5,Ed_bands+IF+A_top3. 金额带 + ae + Family A Top-3,6,36,"is_amount_1_30, is_amount_75_110, if_oof_score...",0.8562,0.0303,0.8520,0.8807,0.8252,55,86,-0.0008



--- 与 6.4a LightGBM 对照（同名组合 AP 应一致）---


,特征组合,AUC-PR_mean,Δ AUC-PR vs BASE
3,Ed_bands. BASE + 两档难样本金额带,0.8599,0.0027



--- 与 6.4a XGBoost 对照（同名组合 AP 应一致）---


,特征组合,AUC-PR_mean,Δ AUC-PR vs BASE
25,Ed_bands. BASE + 两档难样本金额带,0.8563,-0.0006


XGBoost就用Ed_bands+A_top2，比较符合直觉  
LightGBM用Ed_cur=['hours_since_start','is_one_euro']+A_top2


## 6.4a′ 换种子复跑（敏感性检查）

**前置：** §1（`eval_fe_combo`、`cross_val_eval`）、§6.0–6.1（`df_fe`）、§6.4a（`COMBO_SPECS` / `build_fe_combo_specs`）。

用 **`CV_RANDOM_STATE_ALT`** 重跑完整 6.4a 矩阵；结果写入 `combo_lgb_alt` / `combo_xgb_alt`，**不覆盖** 种子 42 的 `combo_lgb` / `combo_xgb`。


In [40]:
# =============================================================================
# 6.4a′ 换随机种子复跑完整组合矩阵（镜像 6.4a，LGB / XGB 分表）
# =============================================================================
from IPython.display import display

CV_RANDOM_STATE_ALT = 123  # 原 6.4a 默认 CV_RANDOM_STATE = 42

_required = ('df_fe', 'eval_fe_combo', 'build_fe_combo_specs', 'FE_FAMILY_A', '_display_combo_table')
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(f'请先运行 §1、§6.0–6.1、§6.4a（至少到 build_fe_combo_specs），缺少: {_missing}')

_combo_specs = COMBO_SPECS if 'COMBO_SPECS' in globals() else build_fe_combo_specs(FE_FAMILY_A)
print(f'6.4a′ 种子={CV_RANDOM_STATE_ALT}（对照：CV_RANDOM_STATE={CV_RANDOM_STATE}）')
print(f'共 {len(_combo_specs)} 种组合 × {CV_N_SPLITS}-fold CV × 2 模型\n')


def run_fe_combo_matrix_seed(
    data: pd.DataFrame,
    combo_specs: list[tuple[list, str]],
    model_name: str = 'LightGBM',
    random_state: int = CV_RANDOM_STATE_ALT,
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []
    n = len(combo_specs)
    for i, (extra, label) in enumerate(combo_specs, start=1):
        if verbose:
            print(f'  [{model_name}] CV {i}/{n}: {label}')
        rows.append(eval_fe_combo(
            data, extra, label, model_name=model_name, random_state=random_state,
        ))
    df_out = pd.DataFrame(rows)
    base_ap = float(df_out.loc[df_out['特征组合'].str.startswith('0.'), 'AUC-PR_mean'].iloc[0])
    df_out['Δ AUC-PR vs BASE'] = df_out['AUC-PR_mean'] - base_ap
    return df_out.sort_values('AUC-PR_mean', ascending=False).reset_index(drop=True)


print('=== 6.4a′ LightGBM（alt seed）===')
combo_lgb_alt = run_fe_combo_matrix_seed(df_fe, _combo_specs, model_name='LightGBM')
_display_combo_table(combo_lgb_alt, '')

print(f'\n=== 6.4a′ XGBoost（alt seed）===')
combo_xgb_alt = run_fe_combo_matrix_seed(df_fe, _combo_specs, model_name='XGBoost')
_display_combo_table(combo_xgb_alt, '')

print('\n--- LightGBM alt：CV Δ>0 的前 3 名 ---')
display(combo_lgb_alt.loc[combo_lgb_alt['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

print('--- XGBoost alt：CV Δ>0 的前 3 名 ---')
display(combo_xgb_alt.loc[combo_xgb_alt['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

# combo_dual（alt seed）
_combo_lgb_idx = combo_lgb_alt.set_index('特征组合')
_combo_xgb_idx = combo_xgb_alt.set_index('特征组合')
_common = _combo_lgb_idx.index.intersection(_combo_xgb_idx.index)
combo_dual_alt = pd.DataFrame([
    {
        '特征组合': label,
        '增量列': _combo_lgb_idx.loc[label, '增量列'],
        'Δ_LGB': float(_combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE']),
        'Δ_XGB': float(_combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE']),
        'Δ_mean': (
            float(_combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
            + float(_combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
        ) / 2,
        '双模型Δ均为正': (
            float(_combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE']) > 0
            and float(_combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE']) > 0
        ),
    }
    for label in _common
]).sort_values('Δ_mean', ascending=False).reset_index(drop=True)

_eligible_alt = combo_dual_alt[combo_dual_alt['双模型Δ均为正']].sort_values('Δ_mean', ascending=False)
print('\n=== 6.4a′ 双模型 Δ 均为正 — Δ_mean Top 3 ===')
display(_eligible_alt.head(3).drop(columns=['增量列']).round(4) if not _eligible_alt.empty else _eligible_alt)

# 与种子 42 对照（若 6.4a 已跑过）
if 'combo_lgb' in globals() and 'combo_xgb' in globals():
    def _seed_compare(ref_lgb, ref_xgb, alt_lgb, alt_xgb, tag: str) -> pd.DataFrame:
        m = ref_lgb[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']].merge(
            alt_lgb[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']],
            on='特征组合', suffixes=(f'_{tag}42', f'_{tag}alt'),
        )
        m[f'ΔAP_{tag}'] = m[f'AUC-PR_mean_{tag}alt'] - m[f'AUC-PR_mean_{tag}42']
        return m.sort_values(f'ΔAP_{tag}', key=abs, ascending=False)

    print(f'\n=== 种子敏感性：|AP_alt − AP_42| 最大的 10 个组合（LightGBM）===')
    _cmp_lgb = _seed_compare(combo_lgb, combo_xgb, combo_lgb_alt, combo_xgb_alt, 'LGB')
    display(_cmp_lgb.head(10).round(4))

    print('=== 种子敏感性：|AP_alt − AP_42| 最大的 10 个组合（XGBoost）===')
    _cmp_xgb = _seed_compare(combo_xgb, combo_lgb, combo_xgb_alt, combo_lgb_alt, 'XGB')
    display(_cmp_xgb.head(10).round(4))

    _watch = [
        '0. 基线（仅 BASE）',
        'Ed_bands. BASE + 两档难样本金额带',
        'A_top3. Family A Top-3（V4+V14+V12）',
        'A1+IF. BASE + if + one_euro_V4（V4）',
        'A1. BASE + one_euro_V4（V4）',
        'IF. BASE + if_oof_score',
    ]
    # Family A Top-3 标签随 TOP_V 排序可能不同，用前缀匹配
    for tbl, name in [(combo_lgb, 'combo_lgb'), (combo_lgb_alt, 'combo_lgb_alt')]:
        _top3 = tbl.loc[tbl['特征组合'].str.startswith('A_top3.'), '特征组合']
        if len(_top3):
            _watch.append(_top3.iloc[0])
    _watch = list(dict.fromkeys(_watch))

    print('\n=== 重点组合：种子 42 vs 123（LGB）===')
    _focus = _cmp_lgb[_cmp_lgb['特征组合'].isin(_watch)].copy()
    display(_focus.round(4))
else:
    print('\n（未找到 combo_lgb/combo_xgb，跳过与种子 42 对照）')


6.4a′ 种子=123（对照：CV_RANDOM_STATE=42）
共 51 种组合 × 5-fold CV × 2 模型

=== 6.4a′ LightGBM（alt seed）===
  [LightGBM] CV 1/51: 0. 基线（仅 BASE）
  [LightGBM] CV 2/51: Ed1. BASE + log1p_amount
  [LightGBM] CV 3/51: Ed2. BASE + hours_since_start
  [LightGBM] CV 4/51: Ed3. BASE + is_micro_testing
  [LightGBM] CV 5/51: Ed4. BASE + is_one_euro
  [LightGBM] CV 6/51: Ed5. BASE + is_amount_1_30
  [LightGBM] CV 7/51: Ed6. BASE + is_amount_75_110
  [LightGBM] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [LightGBM] CV 9/51: Ed_noBands. BASE + EDA 去掉难样本金额带
  [LightGBM] CV 10/51: Ed_bands. BASE + 两档难样本金额带
  [LightGBM] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [LightGBM] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [LightGBM] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [LightGBM] CV 14/51: IF. BASE + if_oof_score
  [LightGBM] CV 15/51: IF+gate. BASE + if_oof_score + if×1EUR
  [LightGBM] CV 16/51: Ed_all+IF. 全部 EDA + if_oof_score
  [LightGBM] CV 17/51: Ed_cur+IF. hours+is_one_euro + if_oof_score
  [LightGBM] C

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A_top3+IF. ae + Family A Top-3,4,34,"if_oof_score, one_euro_V4, one_euro_V14, one_e...",0.8582,0.0283,0.8596,0.8945,0.8272,48,85,0.0079
1,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8582,0.0280,0.8596,0.8945,0.8272,48,85,0.0079
2,FULL. BASE + 全部人为特征（EDA+IF+FamilyA+gate）,14,44,"log1p_amount, hours_since_start, is_micro_test...",0.8568,0.0274,0.8559,0.8938,0.8211,48,88,0.0065
3,A_all+IF. ae + Family A 全族,7,37,"if_oof_score, one_euro_V4, one_euro_V14, one_e...",0.8563,0.0246,0.8541,0.8899,0.8211,50,88,0.0060
4,A6. BASE + one_euro_V26（V26）,1,31,one_euro_V26,0.8557,0.0275,0.8553,0.8901,0.8232,50,87,0.0054
5,A2. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.8557,0.0299,0.8520,0.8877,0.8191,51,89,0.0054
6,A_top2+IF. ae + Family A Top-2,3,33,"if_oof_score, one_euro_V4, one_euro_V14",0.8556,0.0301,0.8535,0.8862,0.8232,52,87,0.0053
7,A4. BASE + one_euro_V16（V16）,1,31,one_euro_V16,0.8555,0.0281,0.8562,0.8921,0.8232,49,87,0.0052
8,Ed1. BASE + log1p_amount,1,31,log1p_amount,0.8551,0.0307,0.8590,0.8980,0.8232,46,87,0.0049
9,Ed6. BASE + is_amount_75_110,1,31,is_amount_75_110,0.8551,0.0310,0.8590,0.8980,0.8232,46,87,0.0048



=== 6.4a′ XGBoost（alt seed）===
  [XGBoost] CV 1/51: 0. 基线（仅 BASE）
  [XGBoost] CV 2/51: Ed1. BASE + log1p_amount
  [XGBoost] CV 3/51: Ed2. BASE + hours_since_start
  [XGBoost] CV 4/51: Ed3. BASE + is_micro_testing
  [XGBoost] CV 5/51: Ed4. BASE + is_one_euro
  [XGBoost] CV 6/51: Ed5. BASE + is_amount_1_30
  [XGBoost] CV 7/51: Ed6. BASE + is_amount_75_110
  [XGBoost] CV 8/51: Ed_all. BASE + 全部 EDA（6 列）
  [XGBoost] CV 9/51: Ed_noBands. BASE + EDA 去掉难样本金额带
  [XGBoost] CV 10/51: Ed_bands. BASE + 两档难样本金额带
  [XGBoost] CV 11/51: Ed_noHours. BASE + EDA 再去掉 hours
  [XGBoost] CV 12/51: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [XGBoost] CV 13/51: Ed_cur. BASE + hours + is_one_euro
  [XGBoost] CV 14/51: IF. BASE + if_oof_score
  [XGBoost] CV 15/51: IF+gate. BASE + if_oof_score + if×1EUR
  [XGBoost] CV 16/51: Ed_all+IF. 全部 EDA + if_oof_score
  [XGBoost] CV 17/51: Ed_cur+IF. hours+is_one_euro + if_oof_score
  [XGBoost] CV 18/51: Ed_noHoursLog+IF. EDA(无金额带/hours/log1p) + AE
  [XGBoost] CV 19/51: IF+log1

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A_all. Family A 全族（6 列）,6,36,"one_euro_V4, one_euro_V14, one_euro_V12, one_e...",0.8600,0.0253,0.8574,0.8923,0.8252,49,86,0.0035
1,Ed_cur+IF+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, if_oof_score, ...",0.8597,0.0285,0.8577,0.8906,0.8272,50,85,0.0032
2,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V4, o...",0.8592,0.0284,0.8568,0.8886,0.8272,51,85,0.0027
3,A6. BASE + one_euro_V26（V26）,1,31,one_euro_V26,0.8589,0.0265,0.8599,0.8928,0.8293,49,84,0.0024
4,Ed_cur+IF. hours+is_one_euro + if_oof_score,3,33,"hours_since_start, is_one_euro, if_oof_score",0.8583,0.0282,0.8592,0.8891,0.8313,51,83,0.0017
5,Ed4. BASE + is_one_euro,1,31,is_one_euro,0.8582,0.0302,0.8599,0.8928,0.8293,49,84,0.0017
6,Ed_all+A_top3. 全部 EDA + Family A Top-3,9,39,"log1p_amount, hours_since_start, is_micro_test...",0.8577,0.0260,0.8583,0.8872,0.8313,52,83,0.0012
7,Ed_noBands. BASE + EDA 去掉难样本金额带,4,34,"log1p_amount, hours_since_start, is_micro_test...",0.8577,0.0295,0.8608,0.8947,0.8293,48,84,0.0012
8,Ed_all+IF+A_top3. 全部 EDA + ae + Family A Top-3,10,40,"log1p_amount, hours_since_start, is_micro_test...",0.8577,0.0278,0.8538,0.8845,0.8252,53,86,0.0012
9,A3. BASE + one_euro_V12（V12）,1,31,one_euro_V12,0.8577,0.0292,0.8638,0.8989,0.8313,46,83,0.0011



--- LightGBM alt：CV Δ>0 的前 3 名 ---


...

,特征组合,AUC-PR_mean_LGB42,Δ AUC-PR vs BASE_LGB42,AUC-PR_mean_LGBalt,Δ AUC-PR vs BASE_LGBalt,ΔAP_LGB
4,Ed_cur+IF. hours+is_one_euro + if_oof_score,0.8599,0.0027,0.8482,-0.0021,-0.0117
3,Ed_bands. BASE + 两档难样本金额带,0.8599,0.0027,0.8496,-0.0007,-0.0104
0,Ed_all+A_top2. 全部 EDA + Family A Top-2,0.8602,0.0030,0.8520,0.0017,-0.0082
5,Ed_cur+IF+A_top2. curated + ae + Family A Top-2,0.8594,0.0022,0.8516,0.0013,-0.0079
23,0. 基线（仅 BASE）,0.8573,0.0000,0.8503,0.0000,-0.0070
20,Ed_noHoursLog+IF. EDA(无金额带/hours/log1p) + AE,0.8578,0.0005,0.8509,0.0007,-0.0068
11,Ed_noHoursLog. BASE + EDA 再去掉 log1p,0.8588,0.0016,0.8522,0.0020,-0.0066
17,IF+hours_since_start. if_oof_score + hours_sin...,0.8581,0.0009,0.8516,0.0013,-0.0066
16,Ed_cur+IF+A_top3. curated + ae + Family A Top-3,0.8582,0.0010,0.8519,0.0016,-0.0063
10,A2+IF. BASE + if + one_euro_V14（V14）,0.8589,0.0016,0.8526,0.0023,-0.0062


=== 种子敏感性：|AP_alt − AP_42| 最大的 10 个组合（XGBoost）===


,特征组合,AUC-PR_mean_XGB42,Δ AUC-PR vs BASE_XGB42,AUC-PR_mean_XGBalt,Δ AUC-PR vs BASE_XGBalt,ΔAP_XGB
50,A_all+IF. ae + Family A 全族,0.8527,-0.0043,0.8576,0.0011,0.0049
6,A2. BASE + one_euro_V14（V14）,0.8574,0.0005,0.8537,-0.0028,-0.0037
1,A_top2. Family A Top-2（V4+V14）,0.8588,0.0019,0.8555,-0.0010,-0.0033
2,A5. BASE + one_euro_V7（V7）,0.8587,0.0018,0.8555,-0.0010,-0.0032
0,A_top3. Family A Top-3（V4+V14+V12）,0.8600,0.0030,0.8570,0.0005,-0.0030
11,A_all. Family A 全族（6 列）,0.8571,0.0001,0.8600,0.0035,0.0029
45,A1+IF. BASE + if + one_euro_V4（V4）,0.8543,-0.0026,0.8571,0.0006,0.0028
48,IF+is_amount_1_30. if_oof_score + is_amount_1_30,0.8536,-0.0034,0.8563,-0.0002,0.0028
44,Ed_cur+IF+A_top3. curated + ae + Family A Top-3,0.8545,-0.0025,0.8572,0.0007,0.0028
31,Ed_cur+IF. hours+is_one_euro + if_oof_score,0.8560,-0.0010,0.8583,0.0017,0.0023



=== 重点组合：种子 42 vs 123（LGB）===


,特征组合,AUC-PR_mean_LGB42,Δ AUC-PR vs BASE_LGB42,AUC-PR_mean_LGBalt,Δ AUC-PR vs BASE_LGBalt,ΔAP_LGB
3,Ed_bands. BASE + 两档难样本金额带,0.8599,0.0027,0.8496,-0.0007,-0.0104
23,0. 基线（仅 BASE）,0.8573,0.0000,0.8503,0.0000,-0.0070
7,IF. BASE + if_oof_score,0.8591,0.0018,0.8534,0.0031,-0.0057
29,A1+IF. BASE + if + one_euro_V4（V4）,0.8567,-0.0006,0.8521,0.0018,-0.0046
13,A_top3. Family A Top-3（V4+V14+V12）,0.8583,0.0010,0.8549,0.0046,-0.0034
50,A1. BASE + one_euro_V4（V4）,0.8536,-0.0036,0.8545,0.0042,0.0009


## 6.4d 补测：`Ed_bands + A_top`（alt seed，重点 XGB）

6.4a / 6.4a′ 的 51 组合**不含** `Ed_bands+A_top*`；6.4c 仅在种子 42 下测过。本 cell 用 **`CV_RANDOM_STATE_ALT`** 补跑，并与种子 42 对照。


In [41]:
# =============================================================================
# 6.4d Ed_bands + A_top2/3 — alt seed（重点 XGBoost）
# =============================================================================
from IPython.display import display

_required = ('df_fe', 'eval_fe_combo', 'AMOUNT_BAND_FEATURES', 'CROSS_FAMILY_A')
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(f'请先运行 §1、§6.0–6.1，缺少: {_missing}')

_rs_alt = CV_RANDOM_STATE_ALT if 'CV_RANDOM_STATE_ALT' in globals() else 123
_rs_base = CV_RANDOM_STATE if 'CV_RANDOM_STATE' in globals() else 42

_bands = list(AMOUNT_BAND_FEATURES)
_family_a = list(CROSS_FAMILY_A)
BAND_ATOP_SPECS = [
    (_bands + _family_a[:2], 'Ed_bands+A_top2. 金额带 + Family A Top-2'),
    (_bands + _family_a[:3], 'Ed_bands+A_top3. 金额带 + Family A Top-3'),
]

print(f'Ed_bands+A_top 补测：种子 42 vs {_rs_alt}（{CV_N_SPLITS}-fold CV）\n')


def _base_ap_for_model(data, model_name: str, random_state: int) -> float:
    rs_default = CV_RANDOM_STATE if 'CV_RANDOM_STATE' in globals() else 42
    tbl_map = {
        (rs_default, 'LightGBM'): 'combo_lgb',
        (rs_default, 'XGBoost'): 'combo_xgb',
        (_rs_alt, 'LightGBM'): 'combo_lgb_alt',
        (_rs_alt, 'XGBoost'): 'combo_xgb_alt',
    }
    tbl = tbl_map.get((random_state, model_name))
    if tbl and tbl in globals():
        _b = globals()[tbl].loc[globals()[tbl]['特征组合'].str.startswith('0.'), 'AUC-PR_mean']
        if len(_b):
            return float(_b.iloc[0])
    return float(eval_fe_combo(
        data, [], '0. 基线（仅 BASE）', model_name=model_name, random_state=random_state,
    )['AUC-PR_mean'])


def _eval_band_atop_matrix(data, specs, model_name: str, random_state: int) -> pd.DataFrame:
    rows = [eval_fe_combo(data, extra, label, model_name=model_name, random_state=random_state)
            for extra, label in specs]
    out = pd.DataFrame(rows)
    base_ap = _base_ap_for_model(data, model_name, random_state)
    out['Δ AUC-PR vs BASE'] = out['AUC-PR_mean'] - base_ap
    out['cv_seed'] = random_state
    return out


# --- XGBoost（重点）---
print('=== XGBoost | 种子 42 ===')
band_atop_xgb_42 = _eval_band_atop_matrix(df_fe, BAND_ATOP_SPECS, 'XGBoost', _rs_base)
display(band_atop_xgb_42.drop(columns=['增量列'], errors='ignore').round(4))

print(f'\n=== XGBoost | 种子 {_rs_alt} ===')
band_atop_xgb_alt = _eval_band_atop_matrix(df_fe, BAND_ATOP_SPECS, 'XGBoost', _rs_alt)
display(band_atop_xgb_alt.drop(columns=['增量列'], errors='ignore').round(4))

_cmp_xgb_ba = band_atop_xgb_42[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']].merge(
    band_atop_xgb_alt[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']],
    on='特征组合', suffixes=('_seed42', f'_seed{_rs_alt}'),
)
_cmp_xgb_ba['ΔAP(alt−42)'] = _cmp_xgb_ba[f'AUC-PR_mean_seed{_rs_alt}'] - _cmp_xgb_ba['AUC-PR_mean_seed42']
print('\n=== XGBoost Ed_bands+A_top：种子 42 vs alt ===')
display(_cmp_xgb_ba.round(4))

# --- LightGBM（对照）---
print('\n=== LightGBM | 种子 42 ===')
band_atop_lgb_42 = _eval_band_atop_matrix(df_fe, BAND_ATOP_SPECS, 'LightGBM', _rs_base)
display(band_atop_lgb_42.drop(columns=['增量列'], errors='ignore').round(4))

print(f'\n=== LightGBM | 种子 {_rs_alt} ===')
band_atop_lgb_alt = _eval_band_atop_matrix(df_fe, BAND_ATOP_SPECS, 'LightGBM', _rs_alt)
display(band_atop_lgb_alt.drop(columns=['增量列'], errors='ignore').round(4))

# 与 6.4c quick 结果对照（若已跑过且为种子 42）
if 'quick_xgb' in globals():
    _q = quick_xgb[quick_xgb['特征组合'].str.startswith('Ed_bands+A_top')]
    if not _q.empty:
        print('\n--- 与 6.4c quick_xgb（种子 42）对照 ---')
        display(_q[['特征组合', 'AUC-PR_mean', 'Δ AUC-PR vs BASE']].round(4))


Ed_bands+A_top 补测：种子 42 vs 123（5-fold CV）

=== XGBoost | 种子 42 ===


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed
0,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8603,0.0275,0.8577,0.8906,0.8272,50,85,0.0034,42
1,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8579,0.0274,0.8599,0.8928,0.8293,49,84,0.0010,42



=== XGBoost | 种子 123 ===


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed
0,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8581,0.0280,0.8550,0.8848,0.8272,53,85,0.0016,123
1,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8570,0.0277,0.8608,0.8947,0.8293,48,84,0.0005,123



=== XGBoost Ed_bands+A_top：种子 42 vs alt ===


...

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed
0,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8588,0.0273,0.8617,0.8967,0.8293,47,84,0.0015,42
1,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8557,0.0271,0.8581,0.8960,0.8232,47,87,-0.0015,42



=== LightGBM | 种子 123 ===


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed
0,Ed_bands+A_top2. 金额带 + Family A Top-2,4,34,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8555,0.0287,0.8556,0.8884,0.8252,51,86,0.0052,123
1,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V4,...",0.8535,0.0313,0.8571,0.8940,0.8232,48,87,0.0032,123



--- 与 6.4c quick_xgb（种子 42）对照 ---


,特征组合,AUC-PR_mean,Δ AUC-PR vs BASE
0,Ed_bands+A_top2. 金额带 + Family A Top-2,0.8603,0.0034
3,Ed_bands+A_top3. 金额带 + Family A Top-3,0.8579,0.0010


## 6.4e 补测：IF × log1p × A_top2 定制组合（双种子）

**前置：** §1–§6.4a（`df_fe`、`if_oof_score`、`CROSS_FAMILY_A`、`eval_fe_combo` 已就绪）。

主矩阵 **不含** 下列三条（均额外加入 `log1p_amount`）：

| 标签 | 增量列 |
|------|--------|
| `IF+Ed_bands+log1p+A_top2` | `if_oof_score` + 两档金额带 + `log1p_amount` + Family A Top-2 |
| `IF+hours+log1p+A_top2` | `if_oof_score` + `hours_since_start` + `log1p_amount` + A_top2 |
| `IF+Ed_cur+log1p+A_top2` | `if_oof_score` + `hours_since_start` + `is_one_euro` + `log1p_amount` + A_top2 |

分别用 **种子 42 / 123** 跑 **StratifiedKFold** CV；**不重跑** 上方 6.4a 全矩阵。

In [42]:
# =============================================================================
# 6.4e IF × log1p × A_top2 定制组合 — 双种子补测（LGB / XGB）
# =============================================================================
import json
from IPython.display import display

CUSTOM_IF_SEEDS = (42, 123)

_required = (
    'df_fe', 'eval_fe_combo', 'make_classifier', 'AMOUNT_BAND_FEATURES', 'CROSS_FAMILY_A',
)
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(f'请先运行 §1–§6.4a，缺少: {_missing}')

_bands = list(AMOUNT_BAND_FEATURES)
_family_a = list(CROSS_FAMILY_A)
_a_top2 = _family_a[:2]
_fe_if = ['if_oof_score']

CUSTOM_IF_SPECS = [
    (
        _fe_if + _bands + ['log1p_amount'] + _a_top2,
        'IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + Family A Top-2',
    ),
    (
        _fe_if + ['hours_since_start', 'log1p_amount'] + _a_top2,
        'IF+hours+log1p+A_top2. IF + hours + log1p + Family A Top-2',
    ),
    (
        _fe_if + ['hours_since_start', 'is_one_euro', 'log1p_amount'] + _a_top2,
        'IF+Ed_cur+log1p+A_top2. IF + hours + is_one_euro + log1p + Family A Top-2',
    ),
]

print('Family A Top-2:', _a_top2)
print('补测组合（相对主矩阵新增 log1p_amount）：')
for extra, label in CUSTOM_IF_SPECS:
    print(f'  • {label}')
    print(f'    增量: {extra}')

_orig_make_classifier = make_classifier


def _make_classifier_for_seed(model_name: str, y_train: pd.Series, seed: int, params: dict | None = None):
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight='balanced', random_state=seed, verbose=-1,
        )
        defaults.update(params)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=seed, eval_metric='logloss', verbosity=0,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    return xgb.XGBClassifier(**defaults)


def _base_ap_for_seed(data, model_name: str, seed: int) -> float:
    tbl_map = {
        (42, 'LightGBM'): 'combo_lgb',
        (42, 'XGBoost'): 'combo_xgb',
        (123, 'LightGBM'): 'combo_lgb_alt',
        (123, 'XGBoost'): 'combo_xgb_alt',
    }
    tbl = tbl_map.get((seed, model_name))
    if tbl and tbl in globals():
        _b = globals()[tbl].loc[globals()[tbl]['特征组合'].str.startswith('0.'), 'AUC-PR_mean']
        if len(_b):
            return float(_b.iloc[0])
    return float(
        eval_fe_combo(data, [], '0. 基线（仅 BASE）', model_name=model_name, random_state=seed)['AUC-PR_mean']
    )


def _eval_custom_if_matrix(data, specs, model_name: str, seed: int) -> pd.DataFrame:
    global make_classifier
    make_classifier = lambda mn, yt, params=None, s=seed: _make_classifier_for_seed(mn, yt, s, params)
    try:
        rows = [
            eval_fe_combo(data, extra, label, model_name=model_name, random_state=seed)
            for extra, label in specs
        ]
    finally:
        make_classifier = _orig_make_classifier
    out = pd.DataFrame(rows)
    base_ap = _base_ap_for_seed(data, model_name, seed)
    out['Δ AUC-PR vs BASE'] = out['AUC-PR_mean'] - base_ap
    out['cv_seed'] = seed
    out['model'] = model_name
    return out


custom_if_lgb_by_seed = {}
custom_if_xgb_by_seed = {}

for seed in CUSTOM_IF_SEEDS:
    sep = '=' * 72
    print(f'\n{sep}\n种子 {seed} | StratifiedKFold {CV_N_SPLITS}-fold\n{sep}')
    custom_if_lgb_by_seed[seed] = _eval_custom_if_matrix(df_fe, CUSTOM_IF_SPECS, 'LightGBM', seed)
    custom_if_xgb_by_seed[seed] = _eval_custom_if_matrix(df_fe, CUSTOM_IF_SPECS, 'XGBoost', seed)

    print(f'\n=== LightGBM | seed={seed} ===')
    display(custom_if_lgb_by_seed[seed].drop(columns=['增量列'], errors='ignore').round(4))
    print(f'\n=== XGBoost | seed={seed} ===')
    display(custom_if_xgb_by_seed[seed].drop(columns=['增量列'], errors='ignore').round(4))

custom_if_lgb_all = pd.concat([custom_if_lgb_by_seed[s] for s in CUSTOM_IF_SEEDS], ignore_index=True)
custom_if_xgb_all = pd.concat([custom_if_xgb_by_seed[s] for s in CUSTOM_IF_SEEDS], ignore_index=True)


def _seed_compare_custom(ref_df: pd.DataFrame, alt_df: pd.DataFrame, alt_seed: int) -> pd.DataFrame:
    key = '特征组合'
    m = ref_df[[key, 'AUC-PR_mean', 'Δ AUC-PR vs BASE', 'FP', 'FN']].merge(
        alt_df[[key, 'AUC-PR_mean', 'Δ AUC-PR vs BASE', 'FP', 'FN']],
        on=key, suffixes=('_seed42', f'_seed{alt_seed}'),
    )
    m[f'ΔAP({alt_seed}−42)'] = m[f'AUC-PR_mean_seed{alt_seed}'] - m['AUC-PR_mean_seed42']
    m['Δ_diff'] = m[f'Δ AUC-PR vs BASE_seed{alt_seed}'] - m['Δ AUC-PR vs BASE_seed42']
    m['符号翻转'] = (m['Δ AUC-PR vs BASE_seed42'] * m[f'Δ AUC-PR vs BASE_seed{alt_seed}'] < 0)
    return m.sort_values(f'ΔAP({alt_seed}−42)', ascending=False)


if 123 in CUSTOM_IF_SEEDS:
    cmp_custom_lgb = _seed_compare_custom(custom_if_lgb_by_seed[42], custom_if_lgb_by_seed[123], 123)
    cmp_custom_xgb = _seed_compare_custom(custom_if_xgb_by_seed[42], custom_if_xgb_by_seed[123], 123)
    print('\n=== LightGBM：定制组合 seed 42 vs 123 ===')
    display(cmp_custom_lgb.round(4))
    print('\n=== XGBoost：定制组合 seed 42 vs 123 ===')
    display(cmp_custom_xgb.round(4))

    _flips = pd.concat([
        cmp_custom_lgb.loc[cmp_custom_lgb['符号翻转'], ['特征组合']].assign(模型='LightGBM'),
        cmp_custom_xgb.loc[cmp_custom_xgb['符号翻转'], ['特征组合']].assign(模型='XGBoost'),
    ], ignore_index=True)
    if not _flips.empty:
        print('\n=== Δ 符号翻转（42 与 123 一正一负）===')
        display(_flips)

export_custom_path = 'COMBO_CUSTOM_IF_log1p_atop2_stratifiedif_seed42_123.json'


def _df_records(df: pd.DataFrame) -> list:
    out = df.copy()
    if '增量列' in out.columns:
        out = out.drop(columns=['增量列'])
    return json.loads(out.round(6).to_json(orient='records', force_ascii=False))

payload = {
    'cv_method': 'StratifiedKFold',
    'cv_n_splits': CV_N_SPLITS,
    'seeds': list(CUSTOM_IF_SEEDS),
    'family_a_top2': _a_top2,
    'specs': [{'label': lbl, 'extra_cols': extra} for extra, lbl in CUSTOM_IF_SPECS],
    'custom_if_lgb': _df_records(custom_if_lgb_all),
    'custom_if_xgb': _df_records(custom_if_xgb_all),
}
if 123 in CUSTOM_IF_SEEDS:
    payload['seed_compare_lgb'] = _df_records(cmp_custom_lgb)
    payload['seed_compare_xgb'] = _df_records(cmp_custom_xgb)

with open(export_custom_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f'\n已导出 → {export_custom_path}')

Family A Top-2: ['one_euro_V4', 'one_euro_V14']
补测组合（相对主矩阵新增 log1p_amount）：
  • IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + Family A Top-2
    增量: ['if_oof_score', 'is_amount_1_30', 'is_amount_75_110', 'log1p_amount', 'one_euro_V4', 'one_euro_V14']
  • IF+hours+log1p+A_top2. IF + hours + log1p + Family A Top-2
    增量: ['if_oof_score', 'hours_since_start', 'log1p_amount', 'one_euro_V4', 'one_euro_V14']
  • IF+Ed_cur+log1p+A_top2. IF + hours + is_one_euro + log1p + Family A Top-2
    增量: ['if_oof_score', 'hours_since_start', 'is_one_euro', 'log1p_amount', 'one_euro_V4', 'one_euro_V14']

种子 42 | StratifiedKFold 5-fold

=== LightGBM | seed=42 ===


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed,model
0,IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + F...,6,36,"if_oof_score, is_amount_1_30, is_amount_75_110...",0.8575,0.0237,0.8568,0.8886,0.8272,51,85,0.0002,42,LightGBM
1,IF+hours+log1p+A_top2. IF + hours + log1p + Fa...,5,35,"if_oof_score, hours_since_start, log1p_amount,...",0.8612,0.0242,0.8553,0.8901,0.8232,50,87,0.0039,42,LightGBM
2,IF+Ed_cur+log1p+A_top2. IF + hours + is_one_eu...,6,36,"if_oof_score, hours_since_start, is_one_euro, ...",0.8537,0.0268,0.8529,0.8896,0.8191,50,89,-0.0036,42,LightGBM



=== XGBoost | seed=42 ===


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE,cv_seed,model
0,IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + F...,6,36,"if_oof_score, is_amount_1_30, is_amount_75_110...",0.8555,0.0301,0.8544,0.8882,0.8232,51,87,-0.0014,42,XGBoost
1,IF+hours+log1p+A_top2. IF + hours + log1p + Fa...,5,35,"if_oof_score, hours_since_start, log1p_amount,...",0.8585,0.0286,0.8535,0.8862,0.8232,52,87,0.0016,42,XGBoost
2,IF+Ed_cur+log1p+A_top2. IF + hours + is_one_eu...,6,36,"if_oof_score, hours_since_start, is_one_euro, ...",0.8549,0.0313,0.8505,0.8821,0.8211,54,88,-0.0021,42,XGBoost



种子 123 | StratifiedKFold 5-fold

=== LightGBM | seed=123 ===


...


=== XGBoost：定制组合 seed 42 vs 123 ===


,特征组合,AUC-PR_mean_seed42,Δ AUC-PR vs BASE_seed42,FP_seed42,FN_seed42,AUC-PR_mean_seed123,Δ AUC-PR vs BASE_seed123,FP_seed123,FN_seed123,ΔAP(123−42),Δ_diff,符号翻转
0,IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + F...,0.8555,-0.0014,51,87,0.8568,0.0002,50,84,0.0012,0.0017,True
2,IF+Ed_cur+log1p+A_top2. IF + hours + is_one_eu...,0.8549,-0.0021,54,88,0.8552,-0.0013,53,84,0.0003,0.0008,False
1,IF+hours+log1p+A_top2. IF + hours + log1p + Fa...,0.8585,0.0016,52,87,0.8576,0.0011,52,83,-0.0009,-0.0005,False



=== Δ 符号翻转（42 与 123 一正一负）===


,特征组合,模型
0,IF+Ed_cur+log1p+A_top2. IF + hours + is_one_eu...,LightGBM
1,IF+Ed_bands+log1p+A_top2. IF + 金额带 + log1p + F...,XGBoost



已导出 → COMBO_CUSTOM_IF_log1p_atop2_stratifiedif_seed42_123.json


IF+hours+log1p+A_top2最好